In [1]:
import numpy as np
from scipy.stats import norm
from scipy.stats import skewnorm
import statsmodels.api as sm

from tqdm import tqdm
import pickle
import pandas as pd
import os

import sys 
sys.path.insert(0, '../src/')

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
import utils
import unet.unet as unet

In [2]:
# path_to_data_test = '/home/paul/Desktop/data_ped_unc/echonet_ped_preprocessed/TEST/'
path_to_data_test = '/home/paul/Desktop/data_ped_unc/echonet_ped_preprocessed/TRAIN/'


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def load_model(ckpt_path):
    model = unet.UNet(1, 2).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt.get("state_dict", ckpt)
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("model."): k = k[len("model."):]
        if k.startswith("net."): k = k[len("net."):]
        new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
    model.eval()
    return model

In [4]:
ckpt_files = [
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_0/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_1/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_2/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_3/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_4/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_5/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_6/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_7/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_8/checkpoints/epoch=49-step=12200.ckpt',
    '/home/paul/Desktop/data_ped_unc/models_eval/ens/combined_no_aug/version_9/checkpoints/epoch=49-step=12200.ckpt'
]

In [5]:
import h5py

# Load all ensemble members
print(f"Loading {len(ckpt_files)} ensemble models...")
models = [load_model(p) for p in ckpt_files]
print("All models loaded.")

Loading 10 ensemble models...
All models loaded.


In [6]:
# Output folder — h5 files will be written here with the same filename as the input
path_to_inference_out = '/home/paul/Desktop/data_ped_unc/predictions/ens/train/'
os.makedirs(path_to_inference_out, exist_ok=True)

In [7]:
file_names = sorted(os.listdir(path_to_data_test))
print(f"Running inference on {len(file_names)} files...")

for file in tqdm(file_names):
    full_filepath = os.path.join(path_to_data_test, file)
    out_filepath  = os.path.join(path_to_inference_out, file)

    with h5py.File(full_filepath, 'r') as src, h5py.File(out_filepath, 'w') as dst:
        phases_present = [p for p in ['ed', 'es'] if p in src]

        for phase in phases_present:
            img = src[phase]['image'][()]  # (H, W)
            img_tensor = torch.Tensor(img).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, H, W)

            # Collect logits from each ensemble member → (n_models, n_classes, H, W)
            logits_list = []
            with torch.no_grad():
                for m in models:
                    logits = m(img_tensor)          # (1, n_classes, H, W)
                    logits_list.append(logits.squeeze(0).cpu().numpy())  # (n_classes, H, W)

            logits_arr = np.stack(logits_list, axis=0)  # (n_samples, n_classes, H, W)

            grp = dst.require_group(phase)
            grp.create_dataset('logits', data=logits_arr)

print("Done. Logits saved to", path_to_inference_out)

Running inference on 2285 files...


100%|██████████| 2285/2285 [01:41<00:00, 22.47it/s]

Done. Logits saved to /home/paul/Desktop/data_ped_unc/predictions/ens/train/
